# NB08a – Saisonale Animationen
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Erzeugt die saisonalen Animationen (GIFs) für NB05 Business Strategy.  
**Dieses Notebook erzeugt — NB05 zeigt.**

Outputs: `output/kuer/<SZ_AKTIV>/animation/anim_A_*.gif`, `anim_B_4panel.gif`, `anim_C_spread.gif`

**Voraussetzungen:**
| Datei | Erzeugt in |
|-------|------------|
| `ch_spot_prices_clean.csv` | NB02 |
| `ch_netzlast_raw.csv` | NB01 |
| `ch_import_export_analyse.csv` | NB07 (optional) |

---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB05 Business Strategy](05_Business_Strategy.ipynb) |
|:---|---:|


---
## 0. Setup

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json as _json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

with open('config.json') as _f:
    CFG = _json.load(_f)

SZ_AKTIV     = CFG['szenarien']['gleichzeitigkeit_aktiv']
DIR_RAW      = os.path.join('data', 'raw')
DIR_PROC     = os.path.join('data', 'processed')
DIR_INTER    = os.path.join('data', 'intermediate')
DIR_INTER_SZ = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR   = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI      = CFG['visualisierung']['output_dpi']       # SSOT: config.json
ANIM_DPI = CFG['visualisierung']['animation_dpi']     # SSOT: config.json

# ── Farben & Stil aus config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt
import matplotlib as _mpl
_mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

print(f"Setup OK | Szenario={SZ_AKTIV} | Animationen → {CHARTS_DIR}")

---
## 1. Daten laden

In [ ]:
# ── Preise und Last für Aggregation laden ────────────────────────────────────
CLEAN_FILE = os.path.join(DIR_PROC, 'ch_spot_prices_clean.csv')
LOAD_FILE  = os.path.join(DIR_RAW,  'ch_netzlast_raw.csv')

if not os.path.exists(CLEAN_FILE):
    print(f'⚠️  {CLEAN_FILE} fehlt → NB01/NB02 zuerst ausführen.')
    df_prices = None
elif not os.path.exists(LOAD_FILE):
    print(f'⚠️  {LOAD_FILE} fehlt → NB01 zuerst ausführen.')
    df_prices = None
else:
    df_prices = pd.read_csv(CLEAN_FILE, parse_dates=['timestamp'])
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices['hour'] = df_prices['timestamp'].dt.hour

    df_load = pd.read_csv(LOAD_FILE, parse_dates=['timestamp'])
    df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)
    df_load['hour'] = df_load['timestamp'].dt.hour

    print(f'df_prices : {df_prices.shape}')
    print(f'df_load   : {df_load.shape}')


---
## 2. Aggregation und Animations-Setup

In [ ]:
if df_prices is None:
    print('⚠️  Daten fehlen — Setup übersprungen.')
    wh_price = wh_load = None
    CROSS_AVAIL = False
    WEEKS = []
    TAGESZEITEN = [0, 7, 12, 19]
else:
    # ── Saisonale Animationen: Setup ─────────────────────────────────────────────
    if df_prices is None:
        print('⚠️  Daten fehlen — Animationssetup übersprungen.')
        wh_price = wh_load = None
        CROSS_AVAIL = False
        WEEKS = []
        TAGESZEITEN = [0, 7, 12, 19]
    else:
        # ── Saisonale Animationen: Setup ─────────────────────────────────────────────
        from matplotlib.animation import FuncAnimation, PillowWriter
        import matplotlib.ticker as mticker
        import matplotlib.patches as mpatches
        
        
        # 52 Kalenderwochen-Aggregate aus 2 Jahren (KW 1-52, gemittelt über beide Jahre)
        df_prices['week'] = df_prices['timestamp'].dt.isocalendar().week.astype(int)
        df_load['week']   = df_load['timestamp'].dt.isocalendar().week.astype(int)
        
        # Pro KW + Stunde: Ø Preis, Ø Last, Ø Import/Export
        # Hinweis: NaN-Werte entstehen bei echten Datenlücken in den ENTSO-E-Rohdaten
        # (z.B. API-Ausfälle, Fehlstunden). dropna() vor Aggregation verhindert dass
        # Wochen mit vollständig fehlenden Stunden NaN in mean/p25/p75 produzieren.
        # Wochen mit NaN-Aggregat werden in den Animationen still übersprungen.
        wh_price = (df_prices.dropna(subset=['price_eur_mwh'])
                    .groupby(['week','hour'])['price_eur_mwh']
                    .agg(mean='mean',
                         p25=lambda x: x.quantile(0.25),
                         p75=lambda x: x.quantile(0.75))
                    .reset_index())
        wh_load  = (df_load.dropna(subset=['load_gw'])
                    .groupby(['week', df_load['timestamp'].dt.hour])['load_gw']
                    .mean().reset_index())
        wh_load.columns = ['week','hour','load_gw']
        
        # Import/Export falls vorhanden
        CROSS_FILE = os.path.join(DIR_INTER, 'ch_import_export_analyse.csv')  # berechnet in NB07
        if os.path.exists(CROSS_FILE):
            df_cx = pd.read_csv(CROSS_FILE, parse_dates=['timestamp'])
            df_cx['timestamp'] = pd.to_datetime(df_cx['timestamp'], utc=True)
            df_cx['week'] = df_cx['timestamp'].dt.isocalendar().week.astype(int)
            df_cx['hour'] = df_cx['timestamp'].dt.hour
            wh_cross = df_cx.groupby(['week','hour'])['net_export_mw'].mean().reset_index()
            CROSS_AVAIL = True
            print('Import/Export verfügbar')
        else:
            CROSS_AVAIL = False
            print('Import/Export nicht verfügbar – Animationen ohne Grenzflüsse')
        
        WEEKS = list(range(1, 53))
        TAGESZEITEN = [0, 7, 12, 19]   # Stunden für Option A
        
        # Hilfsfunktion: Saison-Farbe aus KW
        def kw_to_season_color(kw):
            if kw <= 8 or kw >= 49:  return C_PRIV  # Winter
            elif kw <= 21:            return C_LOAD  # Frühling
            elif kw <= 34:            return C_SOLAR  # Sommer
            else:                     return C_GRENZWERT  # Herbst
        
        def kw_to_month_label(kw):
            month = min(int((kw - 1) * 12 / 52) + 1, 12)
            return ['Jan','Feb','Mär','Apr','Mai','Jun',
                    'Jul','Aug','Sep','Okt','Nov','Dez'][month-1]
        
        print(f'Animationsordner: {CHARTS_DIR}')
        print(f'Wochen: {len(WEEKS)} | Tageszeiten A: {TAGESZEITEN}')
        
    


---
## 3. Verifikation

In [ ]:
if wh_price is None:
    print('⚠️  Übersprungen.')
else:
    # ── Verifikation ────────────────────────────────────────────────────────────
    if wh_price is None:
        print('⚠️  wh_price nicht verfügbar — Verifikation übersprungen.')
    else:
        # ── Verifikation: Aggregation auf NaN prüfen ────────────────────────────────
        nan_price = wh_price[['mean','p25','p75']].isnull().any(axis=1).sum()
        nan_load  = wh_load['load_gw'].isnull().sum()
        total_kw_h = len(wh_price)
        
        print(f'wh_price: {total_kw_h} KW×Stunden-Kombinationen')
        print(f'  NaN-Zeilen (mean/p25/p75): {nan_price}')
        if nan_price > 0:
            print('  Betroffene KW×Stunden:')
            print(wh_price[wh_price[['mean','p25','p75']].isnull().any(axis=1)]
                  [['week','hour','mean','p25','p75']])
            print('  → Diese Frames werden in den Animationen übersprungen (NaN-Guard).')
        else:
            print('  Keine NaN — vollständige Abdeckung.')
        print(f'wh_load : NaN-Zeilen: {nan_load}')
        wh_price.head(3)
        
    


---
## 4. Option A: Animation je Tageszeit (52 Wochen)

In [ ]:
if wh_price is None or not WEEKS:
    print('⚠️  Daten fehlen — übersprungen.')
else:
    # ── Option A: 4 Animationen je Tageszeit (52 Wochen) ────────────────────────
    # Je eine GIF für 00:00, 07:00, 12:00, 19:00
    # Pro Frame: Preis-Kerze (p25/p75/mean) + Netzlast-Balken + Import/Export
    
    n_tageszeiten = len(TAGESZEITEN)
    for t_idx, stunde in enumerate(TAGESZEITEN, 1):
        fig_a, axes_a = plt.subplots(1, 3 if CROSS_AVAIL else 2,
                                      figsize=(13 if CROSS_AVAIL else 10, 5))
        fig_a.patch.set_facecolor(BG_DARK)
        for ax in axes_a: ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for ax in axes_a:
            for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
    
        ax_p, ax_l = axes_a[0], axes_a[1]
        ax_x = axes_a[2] if CROSS_AVAIL else None
    
        # Jahres-Referenz für Achsen
        all_prices = wh_price[wh_price['hour']==stunde]
        p_min = all_prices['p25'].min() * 0.9
        p_max = all_prices['p75'].max() * 1.1
        l_min = wh_load[wh_load['hour']==stunde]['load_gw'].min() * 0.9
        l_max = wh_load[wh_load['hour']==stunde]['load_gw'].max() * 1.1
    
        # Fortschrittsbalken (alle 52 Wochen als Hintergrundlinie)
        for ax in [ax_p, ax_l]:
            ax.set_xlim(0.5, 52.5)
            # Saison-Hintergründe
            for kw_range, col in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                   ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
                ax.axvspan(kw_range[0]-0.5, kw_range[1]+0.5, alpha=0.3, color=col, zorder=0)
    
        ax_p.set_ylim(p_min, p_max)
        ax_p.set_xlabel('Kalenderwoche', color=C_ACHSE)
        ax_p.set_ylabel('EUR/MWh', color=C_ACHSE)
        ax_p.set_title(f'Spot-Preis um {stunde:02d}:00 Uhr', color='white', fontweight='bold')
        ax_l.set_ylim(l_min, l_max)
        ax_l.set_xlabel('Kalenderwoche', color=C_ACHSE)
        ax_l.set_ylabel('GW', color=C_ACHSE)
        ax_l.set_title(f'Netzlast um {stunde:02d}:00 Uhr', color='white', fontweight='bold')
        ax_l.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f} GW'))
    
        if ax_x:
            x_data = wh_cross[wh_cross['hour']==stunde]
            x_min = x_data['net_export_mw'].min() * 1.1 if x_data['net_export_mw'].min() < 0 else -200
            x_max = x_data['net_export_mw'].max() * 1.1
            ax_x.set_xlim(0.5, 52.5); ax_x.set_ylim(x_min, x_max)
            ax_x.axhline(0, color='white', lw=LW_DUENN, linestyle='--')
            ax_x.set_xlabel('Kalenderwoche', color=C_ACHSE)
            ax_x.set_ylabel('MW', color=C_ACHSE)
            ax_x.set_title(f'Netto-Export um {stunde:02d}:00 Uhr', color='white', fontweight='bold')
            for kw_range, col in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                   ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
                ax_x.axvspan(kw_range[0]-0.5, kw_range[1]+0.5, alpha=0.3, color=col, zorder=0)
    
        # KW-Tick-Labels: Monatsanfänge
        tick_kws = [1,5,9,13,18,22,27,31,35,40,44,48,52]
        for ax in [ax_p, ax_l] + ([ax_x] if ax_x else []):
            ax.set_xticks(tick_kws)
            ax.set_xticklabels([kw_to_month_label(k) for k in tick_kws], fontsize=FS_LEGENDE)
            ax.grid(True, axis='y', alpha=ALPHA_FLAECHE)
    
        title_obj = fig_a.suptitle('', color='white', fontsize=12, fontweight='bold')
        kw_line_p = ax_p.axvline(1, color='white', lw=LW, zorder=5)
        kw_line_l = ax_l.axvline(1, color='white', lw=LW, zorder=5)
        kw_line_x = ax_x.axvline(1, color='white', lw=LW, zorder=5) if ax_x else None
    
        # Aktueller Lastpunkt (Preis wird via errorbar gezeichnet)
        pt_l = ax_l.plot([], [], 'o', color='white', markersize=8, zorder=6)[0]
        if ax_x: pt_x = ax_x.bar([0], [0], 0.8, color='white', alpha=0.8, zorder=6)
    
        # Historische Linie
        hist_p, = ax_p.plot([], [], color='#888', lw=LW_DUENN, alpha=0.5)
        hist_l, = ax_l.plot([], [], color='#888', lw=LW_DUENN, alpha=0.5)
    
        hist_kws, hist_prices, hist_loads = [], [], []
        # Errorbar-Container: wird pro Frame ersetzt (kein collections.clear())
        _eb_container = [None]
    
        def update_a(frame_kw):
            row_p = wh_price[(wh_price['week']==frame_kw) & (wh_price['hour']==stunde)]
            row_l = wh_load[(wh_load['week']==frame_kw) & (wh_load['hour']==stunde)]
            if row_p.empty: return []
            mean_p = float(row_p['mean'])
            p25_v  = float(row_p['p25'])
            p75_v  = float(row_p['p75'])
            load_v = float(row_l['load_gw']) if not row_l.empty else 0
            # NaN-Schutz: bei fehlenden Werten Frame überspringen
            import math
            if any(math.isnan(v) for v in [mean_p, p25_v, p75_v, load_v]): return []
            col    = kw_to_season_color(frame_kw)
    
            hist_kws.append(frame_kw); hist_prices.append(mean_p); hist_loads.append(load_v)
            hist_p.set_data(hist_kws, hist_prices)
            hist_l.set_data(hist_kws, hist_loads)
    
            # Vorherigen Errorbar-Container entfernen
            if _eb_container[0] is not None:
                try:
                    _eb_container[0].remove()
                except Exception:
                    pass
            # Neuen Errorbar zeichnen und Container merken
            lo = max(0.0, mean_p - p25_v)
            hi = max(0.0, p75_v  - mean_p)
            eb = ax_p.errorbar([frame_kw], [mean_p],
                                yerr=[[lo], [hi]],
                                fmt='o', color=col, capsize=5, markersize=8,
                                elinewidth=2, capthick=2, zorder=6)
            _eb_container[0] = eb
            pt_l.set_data([frame_kw], [load_v]); pt_l.set_color(col)
    
            kw_line_p.set_xdata([frame_kw, frame_kw])
            kw_line_l.set_xdata([frame_kw, frame_kw])
    
            if ax_x and CROSS_AVAIL:
                row_x = wh_cross[(wh_cross['week']==frame_kw) & (wh_cross['hour']==stunde)]
                xval  = float(row_x['net_export_mw']) if not row_x.empty else 0
                xc    = C_LOAD if xval >= 0 else C_UTIL
                ax_x.cla()
                ax_x.set_facecolor(BG_PANEL); ax_x.tick_params(colors=C_TICK)
                for sp in ax_x.spines.values(): sp.set_edgecolor(C_SPINE)
                ax_x.set_xlim(0.5,52.5); ax_x.set_ylim(x_min, x_max)
                ax_x.axhline(0, color='white', lw=LW_DUENN, linestyle='--')
                # Historische Balken
                for hw, hx in zip(hist_kws, [float(wh_cross[(wh_cross['week']==w) &
                                  (wh_cross['hour']==stunde)]['net_export_mw'].iloc[0])
                                  if not wh_cross[(wh_cross['week']==w) &
                                  (wh_cross['hour']==stunde)].empty else 0
                                  for w in hist_kws]):
                    hc = '#2E7D32' if hx >= 0 else C_FEED
                    ax_x.bar(hw, hx, 0.7, color=hc, alpha=0.35, zorder=3)
                ax_x.bar(frame_kw, xval, 0.9, color=xc, alpha=0.9, zorder=5)
                ax_x.set_title(f'Netto-Export um {stunde:02d}:00 Uhr', color='white', fontweight='bold')
                ax_x.set_xticks(tick_kws)
                ax_x.set_xticklabels([kw_to_month_label(k) for k in tick_kws], fontsize=FS_LEGENDE)
                ax_x.set_ylabel('MW', color=C_ACHSE)
                ax_x.set_xlabel('Kalenderwoche', color=C_ACHSE)
                ax_x.grid(True, axis='y', alpha=ALPHA_FLAECHE)
                kw_line_x.set_xdata([frame_kw, frame_kw])
    
            season_name = ('Winter' if frame_kw<=8 or frame_kw>=49 else
                           'Frühling' if frame_kw<=21 else
                           'Sommer' if frame_kw<=34 else 'Herbst')
            title_obj.set_text(
                f'Saisonaler Verlauf {stunde:02d}:00 Uhr — KW {frame_kw:02d} '
                f'({kw_to_month_label(frame_kw)}, {season_name})  |  '
                f'Preis: {mean_p:.0f} EUR/MWh  |  Last: {load_v:.2f} GW')
            return []
    
        ani_a = FuncAnimation(fig_a, update_a, frames=WEEKS, interval=200, blit=False)
        fname = os.path.join(CHARTS_DIR, f'kuer_nb08a_anim_A_{stunde:02d}h.gif')
        print(f'  Anim A [{t_idx}/{n_tageszeiten}] {stunde:02d}:00 → speichere...', end=' ', flush=True)
        ani_a.save(fname, writer=PillowWriter(fps=5), dpi=ANIM_DPI)
        print(f'✅ ({os.path.getsize(fname)//1024} KB)')
        plt.close(fig_a)
        hist_kws.clear(); hist_prices.clear(); hist_loads.clear()
    
    print('Option A abgeschlossen (4 Animationen)')
    
    

---
## 5. Option B: 4-Panel-Animation

In [ ]:
if wh_price is None or not WEEKS:
    print('⚠️  Daten fehlen — übersprungen.')
else:
    # ── Option B: 4-Panel-Animation (alle 4 Tageszeiten, 52 Wochen) ─────────────
    # 2×2 Grid: oben Preis, unten Last — Spalten 07:00 und 19:00 (wichtigste Stunden)
    # Alle 4 Tageszeiten: 00:00 / 07:00 / 12:00 / 19:00
    
    STUNDEN_B = [0, 7, 12, 19]
    STUNDEN_LABELS = ['00:00 (Nacht)', '07:00 (Morgen)', '12:00 (Mittag)', '19:00 (Abend)']
    STUNDEN_COLORS = [C_PRIV, C_SOLAR, C_GRENZWERT, C_UTIL]
    
    fig_b, axes_b = plt.subplots(2, 4, figsize=(22, 9))
    fig_b.patch.set_facecolor(BG_DARK)
    for ax in axes_b.flat:
        ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
    
    title_b = fig_b.suptitle('', color='white', fontsize=FS_TITEL, fontweight='bold')
    
    # Achsenlimits vorab
    p_limits = {}
    l_limits = {}
    for s in STUNDEN_B:
        dp = wh_price[wh_price['hour']==s]
        dl = wh_load[wh_load['hour']==s]
        p_limits[s] = (dp['p25'].min()*0.9, dp['p75'].max()*1.1)
        l_limits[s] = (dl['load_gw'].min()*0.95, dl['load_gw'].max()*1.05)
    
    for col_idx, (s, lbl, col) in enumerate(zip(STUNDEN_B, STUNDEN_LABELS, STUNDEN_COLORS)):
        for row_idx in range(2):
            ax = axes_b[row_idx, col_idx]
            ax.set_xlim(0.5, 52.5)
            if row_idx == 0:
                ax.set_ylim(*p_limits[s])
                ax.set_title(lbl, color=col, fontsize=FS_ACHSE, fontweight='bold')
                if col_idx == 0: ax.set_ylabel('EUR/MWh', color=C_ACHSE)
            else:
                ax.set_ylim(*l_limits[s])
                ax.set_xlabel('Monat', color=C_ACHSE)
                ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}'))
                if col_idx == 0: ax.set_ylabel('GW', color=C_ACHSE)
            ax.set_xticks(tick_kws)
            ax.set_xticklabels([kw_to_month_label(k) for k in tick_kws], fontsize=FS_KLEIN)
            ax.grid(True, axis='y', alpha=0.10)
            for kw_range, bg in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                  ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
                ax.axvspan(kw_range[0]-.5, kw_range[1]+.5, alpha=0.25, color=bg, zorder=0)
    
    # Zeile-Labels links
    axes_b[0,0].text(-6, sum(p_limits[0])/2, 'Spot-Preis\n[EUR/MWh]',
                      color='white', fontsize=FS_TICK, ha='center', va='center', rotation=90,
                      transform=axes_b[0,0].transData)
    
    plt.tight_layout(rect=[0,0,1,0.95])
    
    hist_b = {s: {'kws':[], 'prices':[], 'loads':[]} for s in STUNDEN_B}
    
    def update_b(frame_kw):
        col_cur = kw_to_season_color(frame_kw)
        season_name = ('Winter' if frame_kw<=8 or frame_kw>=49 else
                       'Frühling' if frame_kw<=21 else
                       'Sommer' if frame_kw<=34 else 'Herbst')
        title_b.set_text(
            f'Saisonaler Verlauf — KW {frame_kw:02d} ({kw_to_month_label(frame_kw)}, {season_name})')
    
        for col_idx, (s, col_s) in enumerate(zip(STUNDEN_B, STUNDEN_COLORS)):
            row_p = wh_price[(wh_price['week']==frame_kw) & (wh_price['hour']==s)]
            row_l = wh_load[(wh_load['week']==frame_kw) & (wh_load['hour']==s)]
            if row_p.empty: continue
            mean_p = float(row_p['mean'])
            p25_v  = float(row_p['p25'])
            p75_v  = float(row_p['p75'])
            load_v = float(row_l['load_gw']) if not row_l.empty else 0
            import math
            if any(math.isnan(v) for v in [mean_p, p25_v, p75_v, load_v]): continue
            hist_b[s]['kws'].append(frame_kw)
            hist_b[s]['prices'].append(mean_p)
            hist_b[s]['loads'].append(load_v)
    
            ax_price = axes_b[0, col_idx]
            ax_load  = axes_b[1, col_idx]
    
            # Historische Linie
            if len(hist_b[s]['kws']) > 1:
                ax_price.plot(hist_b[s]['kws'], hist_b[s]['prices'],
                              color=col_s, lw=0.9, alpha=0.35, zorder=3)
                ax_load.plot(hist_b[s]['kws'], hist_b[s]['loads'],
                             color=col_s, lw=0.9, alpha=0.35, zorder=3)
    
            # Aktueller Punkt
            # max(0,...): Schutz gegen negative yerr wenn mean ausserhalb p25/p75
            # (tritt auf wenn wöchentlicher Mittelwert und stündliche Quantile
            # unterschiedliche Zeitfenster aggregieren → mean kann < p25 sein)
            ax_price.errorbar([frame_kw], [mean_p],
                               yerr=[[max(0, mean_p-p25_v)],[max(0, p75_v-mean_p)]],
                               fmt='o', color=col_cur, capsize=4,
                               markersize=7, elinewidth=1.8, zorder=6)
            ax_load.plot([frame_kw], [load_v], 'o',
                          color=col_cur, markersize=7, zorder=6)
    
        return []
    
    ani_b = FuncAnimation(fig_b, update_b, frames=WEEKS, interval=200, blit=False)
    fname_b = os.path.join(CHARTS_DIR, 'kuer_nb08a_kuer_nb08a_anim_B_4panel.gif')
    print('  Anim B (4-Panel) → speichere...', end=' ', flush=True)
    ani_b.save(fname_b, writer=PillowWriter(fps=5), dpi=ANIM_DPI)
    print(f'✅ ({os.path.getsize(fname_b)//1024} KB)')
    plt.close(fig_b)
    
    

---
## 6. Option C: Spread-Animation rollierend

In [ ]:
if wh_price is None or not WEEKS:
    print('⚠️  Daten fehlen — übersprungen.')
else:
    # ── Option C: Spread-Animation (52 Wochen rollierend) ────────────────────────
    # Zeigt Arbitrage-Spread (p75-p25 über alle 24h) als aufbauende Kurve
    # + Balken für Negativpreis-Anteil + Dispatch-Effizienz
    
    # Wöchentlicher Spread = p75-Ø minus p25-Ø (über alle Stunden eines Tages)
    week_spread = (df_prices.groupby('week')['price_eur_mwh']
                   .agg(p25=lambda x: x.quantile(0.25),
                        p75=lambda x: x.quantile(0.75),
                        mean='mean',
                        neg_pct=lambda x: (x < 0).mean() * 100)
                   .reset_index())
    week_spread['spread'] = week_spread['p75'] - week_spread['p25']
    
    # Dispatch-Effizienz: Anteil Stunden wo Dispatch sinnvoll wäre
    df_prices['day'] = df_prices['timestamp'].dt.date
    df_prices['p75_day'] = df_prices.groupby('day')['price_eur_mwh'].transform(
        lambda x: x.quantile(0.75))
    df_prices['p25_day'] = df_prices.groupby('day')['price_eur_mwh'].transform(
        lambda x: x.quantile(0.25))
    df_prices['dispatch_load'] = df_prices['price_eur_mwh'] <= df_prices['p25_day']
    df_prices['dispatch_feed'] = df_prices['price_eur_mwh'] >= df_prices['p75_day']
    week_dispatch = (df_prices.groupby('week')[['dispatch_load','dispatch_feed']]
                     .mean() * 100).reset_index()
    week_spread = week_spread.merge(week_dispatch, on='week')
    
    fig_c, axes_c = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    fig_c.patch.set_facecolor(BG_DARK)
    for ax in axes_c:
        ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
        ax.set_xlim(0.5, 52.5)
        ax.grid(True, axis='y', alpha=ALPHA_FLAECHE)
        for kw_range, col_bg in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                  ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
            ax.axvspan(kw_range[0]-.5, kw_range[1]+.5, alpha=0.25, color=col_bg, zorder=0)
    
    axes_c[0].set_ylim(0, week_spread['spread'].max() * 1.15)
    axes_c[0].set_ylabel('EUR/MWh', color=C_ACHSE)
    axes_c[0].set_title('Wöchentlicher Arbitrage-Spread (p75 − p25, Tagesdurchschnitt)',
                        color='white', fontweight='bold')
    axes_c[0].axhline(week_spread['spread'].mean(), color='#888', lw=1, linestyle=':')
    
    axes_c[1].set_ylim(0, week_spread['neg_pct'].max() * 1.2 + 0.1)
    axes_c[1].set_ylabel('%', color=C_ACHSE)
    axes_c[1].set_title('Negativpreis-Anteil [%] — Ladezyklen ohne Kosten möglich',
                        color='white', fontweight='bold')
    
    axes_c[2].set_ylim(0, 30)
    axes_c[2].set_ylabel('%', color=C_ACHSE)
    axes_c[2].set_xlabel('Monat', color=C_ACHSE)
    axes_c[2].set_title('Dispatch-Stunden [%] — Anteil Stunden mit Lade/Einspeisesignal',
                        color='white', fontweight='bold')
    
    axes_c[2].set_xticks(tick_kws)
    axes_c[2].set_xticklabels([kw_to_month_label(k) for k in tick_kws])
    
    title_c = fig_c.suptitle('', color='white', fontsize=FS_TITEL, fontweight='bold')
    
    line_spread, = axes_c[0].plot([], [], color=C_PRICE, lw=LW_DICK, zorder=4)
    line_neg,    = axes_c[1].plot([], [], color=C_UTIL, lw=LW_DICK, zorder=4)
    
    kws_hist, spread_hist, neg_hist = [], [], []
    load_hist, feed_hist = [], []
    
    def update_c(frame_kw):
        row = week_spread[week_spread['week']==frame_kw]
        if row.empty: return []
        sp_val  = float(row['spread'])
        ng_val  = float(row['neg_pct'])
        ld_val  = float(row['dispatch_load']) * 24 / 100  # Stunden
        fd_val  = float(row['dispatch_feed']) * 24 / 100
        col_cur = kw_to_season_color(frame_kw)
    
        kws_hist.append(frame_kw)
        spread_hist.append(sp_val); neg_hist.append(ng_val)
        load_hist.append(ld_val);   feed_hist.append(fd_val)
    
        line_spread.set_data(kws_hist, spread_hist)
        line_neg.set_data(kws_hist, neg_hist)
    
        # Aktuellen Punkt hervorheben
        axes_c[0].plot([frame_kw], [sp_val], 'o', color=col_cur, markersize=8, zorder=6)
        axes_c[1].plot([frame_kw], [ng_val], 'o', color=col_cur, markersize=8, zorder=6)
    
        # Dispatch-Balken: Laden (blau) + Einspeisen (rot) gestapelt
        axes_c[2].bar(frame_kw, ld_val, 0.8, color=C_CHARGE, alpha=0.8, zorder=4)
        axes_c[2].bar(frame_kw, fd_val, 0.8, bottom=ld_val,
                       color=C_FEED, alpha=0.8, zorder=4)
    
        season_name = ('Winter' if frame_kw<=8 or frame_kw>=49 else
                       'Frühling' if frame_kw<=21 else
                       'Sommer' if frame_kw<=34 else 'Herbst')
        title_c.set_text(
            f'Arbitrage-Kennzahlen KW {frame_kw:02d} ({kw_to_month_label(frame_kw)}, '
            f'{season_name})  |  Spread: {sp_val:.0f} EUR/MWh  |  '
            f'Negativ: {ng_val:.1f}%')
        return []
    
    ani_c = FuncAnimation(fig_c, update_c, frames=WEEKS, interval=200, blit=False)
    fname_c = os.path.join(CHARTS_DIR, 'kuer_nb08a_kuer_nb08a_anim_C_spread.gif')
    print('  Anim C (Spread) → speichere...', end=' ', flush=True)
    ani_c.save(fname_c, writer=PillowWriter(fps=5), dpi=ANIM_DPI)
    print(f'✅ ({os.path.getsize(fname_c)//1024} KB)')
    plt.close(fig_c)
    
    # Übersicht
    print('\nAlle saisonalen Animationen:')
    for f in sorted(os.listdir(CHARTS_DIR)):
        if f.startswith('kuer_nb08a') and f.endswith('.gif'):
            kb = os.path.getsize(os.path.join(CHARTS_DIR, f)) / 1024
            print(f'  {f:<45} {kb:.0f} KB')
    
    

---

In [ ]:
# ── Abschlusskontrolle ───────────────────────────────────────────────────────
print("=== Abschlusskontrolle NB08a (Kür) ===")
gifs = [f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_nb08a') and f.endswith('.gif')]
for f in sorted(gifs):
    kb = os.path.getsize(os.path.join(CHARTS_DIR, f)) / 1024
    print(f"  ✅  {f:<45} {kb:.0f} KB")
print(f"  {len(gifs)} GIF(s) erzeugt → {CHARTS_DIR}")
print("=== Ende ===")


---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB05 Business Strategy](05_Business_Strategy.ipynb) |
|:---|---:|
